# Staged Embarrassment Learning — CIFAR-10 Final
**Two separate notebooks in one:**
- Part A (Cells 1-6): Baseline — 30 epochs, full data, no tricks
- Part B (Cells 7-12): Staged system — per-class curriculum, sparse updates, embarrassment signal

**Testing:** 100 images per class = 1000 total test images, real held-out accuracy

**Runtime → T4 GPU before running**

In [ ]:
# CELL 1 — Imports
import torch, time, warnings, random, math
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import pandas as pd
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU   : {torch.cuda.get_device_name(0)}')

torch.manual_seed(42); np.random.seed(42); random.seed(42)

CLASS_NAMES = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']
N_CLASSES   = 10
BATCH_SIZE  = 64
BASE_LR     = 1e-3
TOTAL_PARAMS = None

# Staged config
STAGE_CONFIG = [
    ('Stage 1 Easy',    0.00, 0.25, 0.30, 10),
    ('Stage 2 Medium',  0.20, 0.50, 0.45, 10),
    ('Stage 3 Hard',    0.40, 0.65, 0.58, 10),
    ('Stage 4 Harder',  0.60, 0.82, 0.68, 10),
    ('Stage 5 Hardest', 0.78, 1.00, 0.76, 10),
]
SAMPLES_PER_CLASS = 2000
TEMPERATURE       = 1.5
BASELINE_EPOCHS   = 30

print(f'Classes        : {N_CLASSES}')
print(f'Baseline epochs: {BASELINE_EPOCHS}')
print(f'Staged epochs  : {sum(s[4] for s in STAGE_CONFIG)}')
print(f'Samples/stage  : {SAMPLES_PER_CLASS*N_CLASSES} ({SAMPLES_PER_CLASS}/class)')

In [ ]:
# CELL 2 — Dataset + Test Set (100 per class = 1000 total)
train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.4914,0.4822,0.4465],[0.2470,0.2435,0.2616])
])
test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.4914,0.4822,0.4465],[0.2470,0.2435,0.2616])
])
vis_tf = transforms.Compose([
    transforms.ToTensor()
])

full_train = datasets.CIFAR10('/content/data', train=True,  download=True, transform=train_tf)
full_test  = datasets.CIFAR10('/content/data', train=False, download=True, transform=test_tf)
full_vis   = datasets.CIFAR10('/content/data', train=False, download=False, transform=vis_tf)

# Full test loader (10000 images)
full_test_loader = DataLoader(full_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Held-out test set: exactly 100 per class = 1000 total
# These are NEVER seen during training
test_100_idx = []
for c in range(N_CLASSES):
    class_idx = [i for i,(_, l) in enumerate(full_test) if l == c][:100]
    test_100_idx.extend(class_idx)

test_100_loader = DataLoader(
    Subset(full_test, test_100_idx),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2
)
# Visual loader (no normalize) for thermal snapshots
vis_100_loader = DataLoader(
    Subset(full_vis, test_100_idx),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2
)

# Training loader
base_loader = DataLoader(full_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

print(f'Full train   : {len(full_train)}')
print(f'Full test    : {len(full_test)}')
print(f'Held-out test: {len(test_100_idx)} (100 per class x 10 classes)')

In [ ]:
# CELL 3 — Model + FLOPs
def build_model():
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    m.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()
    m.fc      = nn.Linear(512, N_CLASSES)
    return m.to(DEVICE)

tmp = build_model()
TOTAL_PARAMS = sum(p.numel() for p in tmp.parameters())
del tmp

def flops(sp, n): return 2 * TOTAL_PARAMS * (1.0 - sp) * n

print(f'ResNet-18 params     : {TOTAL_PARAMS:,}')
print(f'FLOPs/epoch baseline : {2*TOTAL_PARAMS*50000/1e12:.3f}T')

In [ ]:
# CELL 4 — Math Functions

def pc_embarrassment(logits, labels, T=TEMPERATURE):
    losses = nn.CrossEntropyLoss(reduction='none')(logits/T, labels)
    E, C   = np.zeros(N_CLASSES), np.zeros(N_CLASSES)
    for c in range(N_CLASSES):
        m = (labels == c)
        if m.sum() > 0:
            e = losses[m].mean().item()
            E[c] = e
            C[c] = max(0.0, 1.0 - e)
    return E, C

def sparse_update(model, gamma):
    tot = guilty = 0
    for p in model.parameters():
        if p.grad is not None:
            mask   = (p.grad.abs() > gamma).float()
            p.grad.mul_(mask)
            tot   += mask.numel()
            guilty += mask.sum().item()
    return 1.0 - (guilty / max(tot, 1))

def evaluate_full(model, loader):
    """Accuracy on full test set."""
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            correct += (model(imgs).argmax(1) == labels).sum().item()
            total   += labels.size(0)
    return correct / total

def evaluate_100_per_class(model):
    """100 images per class test — returns per-class and overall accuracy."""
    model.eval()
    cc, ct = [0]*N_CLASSES, [0]*N_CLASSES
    with torch.no_grad():
        for imgs, labels in test_100_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            preds = model(imgs).argmax(1)
            for i in range(len(labels)):
                l = labels[i].item()
                cc[l] += (preds[i] == labels[i]).item()
                ct[l] += 1
    per_class = np.array([cc[i]/max(ct[i],1) for i in range(N_CLASSES)])
    overall   = sum(cc) / sum(ct)
    return per_class, overall

def get_predictions_on_100(model):
    """Get raw predictions for thermal snapshot visualization."""
    model.eval()
    all_imgs, all_preds, all_labels, all_probs = [], [], [], []
    with torch.no_grad():
        for imgs_vis, labels in vis_100_loader:
            imgs_n = torch.stack([test_tf(transforms.ToPILImage()(img)) for img in imgs_vis])
            imgs_n = imgs_n.to(DEVICE)
            logits = model(imgs_n)
            probs  = torch.softmax(logits, dim=1)
            preds  = logits.argmax(1)
            all_imgs.extend(imgs_vis.numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())
    return all_imgs, all_preds, all_labels, all_probs

print('Math functions ready')

In [ ]:
# CELL 5 — Per-Class Sorted Pools + Auto Guilt Threshold
print('Sorting each class easy to hard...')
scorer  = build_model()
scorer.eval()
crit_nr = nn.CrossEntropyLoss(reduction='none')

all_losses, all_labels_arr = [], []
with torch.no_grad():
    for imgs, labels in DataLoader(full_train, batch_size=256, shuffle=False, num_workers=2):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        losses = crit_nr(scorer(imgs), labels)
        all_losses.extend(losses.cpu().numpy())
        all_labels_arr.extend(labels.cpu().numpy())
del scorer

all_losses     = np.array(all_losses)
all_labels_arr = np.array(all_labels_arr)

class_pools = {}
for c in range(N_CLASSES):
    idx   = np.where(all_labels_arr == c)[0]
    order = np.argsort(all_losses[idx])
    class_pools[c] = idx[order].tolist()
    print(f'  {CLASS_NAMES[c]:<12}: easy={all_losses[idx[order[0]]]:.3f}  hard={all_losses[idx[order[-1]]]:.3f}')

# Auto guilt threshold p20
tmp_m = build_model()
tmp_o = optim.Adam(tmp_m.parameters(), lr=BASE_LR)
ig, il = next(iter(DataLoader(full_train, batch_size=128)))
tmp_o.zero_grad()
nn.CrossEntropyLoss()(tmp_m(ig.to(DEVICE)), il.to(DEVICE)).backward()
grads = np.concatenate([p.grad.abs().cpu().numpy().flatten()
                        for p in tmp_m.parameters() if p.grad is not None])
GUILT_THRESHOLD = float(np.percentile(grads, 20))
del tmp_m
print(f'\nGuilt threshold (p20): {GUILT_THRESHOLD:.6f}')

def get_stage_loader(si):
    _, ps, pe, _, _ = STAGE_CONFIG[si]
    idx = []
    for c in range(N_CLASSES):
        pool = class_pools[c]
        s    = int(ps * len(pool))
        e    = min(int(pe * len(pool)), s + SAMPLES_PER_CLASS)
        idx += pool[s:e]
    random.shuffle(idx)
    return DataLoader(Subset(full_train, idx), batch_size=BATCH_SIZE,
                      shuffle=True, num_workers=2)

print('\nBalance per stage:')
for si in range(len(STAGE_CONFIG)):
    ld = get_stage_loader(si)
    ct = [0]*N_CLASSES
    for _, lbl in ld:
        for l in lbl: ct[l.item()] += 1
    print(f'  {STAGE_CONFIG[si][0]}: total={sum(ct)}  per_class~{sum(ct)//N_CLASSES}')

In [ ]:
# CELL 6 — Baseline Training (30 epochs, full data, no tricks)
print(f'Baseline: {BASELINE_EPOCHS} epochs on {len(full_train)} images')

b_model = build_model()
b_opt   = optim.Adam(b_model.parameters(), lr=BASE_LR)
b_crit  = nn.CrossEntropyLoss()
b_hist  = []
b_flops = 0.0
torch.cuda.reset_peak_memory_stats()
b_start = time.time()

for ep in range(BASELINE_EPOCHS):
    b_model.train()
    ls = cor = tot = 0
    for imgs, labels in base_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        b_opt.zero_grad()
        out  = b_model(imgs)
        loss = b_crit(out, labels)
        loss.backward()
        b_opt.step()
        ls  += loss.item() * labels.size(0)
        cor += (out.argmax(1) == labels).sum().item()
        tot += labels.size(0)
        b_flops += 2 * TOTAL_PARAMS * labels.size(0)

    # Both accuracies every epoch
    full_acc = evaluate_full(b_model, full_test_loader)
    pc_acc, test100_acc = evaluate_100_per_class(b_model)
    b_hist.append({
        'epoch': ep+1,
        'train_acc': cor/tot,
        'train_loss': ls/tot,
        'test_full_acc': full_acc,
        'test_100_acc': test100_acc,
        'flops_T': b_flops/1e12,
        'time_s': time.time()-b_start,
    })
    for ci, cn in enumerate(CLASS_NAMES):
        b_hist[-1][f'acc_{cn}'] = pc_acc[ci]

    bar = '#'*int(full_acc*20) + '-'*(20-int(full_acc*20))
    print(f'  Ep{ep+1:2d}/{BASELINE_EPOCHS} [{bar}] full={full_acc:.1%}  test100={test100_acc:.1%}  train={cor/tot:.1%}')

b_elapsed   = time.time() - b_start
b_hist_df   = pd.DataFrame(b_hist)
B_FULL_ACC  = evaluate_full(b_model, full_test_loader)
B_PC, B_100 = evaluate_100_per_class(b_model)

print(f'\nBaseline done in {b_elapsed:.0f}s')
print(f'  Full test accuracy  : {B_FULL_ACC:.1%}')
print(f'  100-per-class test  : {B_100:.1%}')
print(f'  Total FLOPs         : {b_flops/1e12:.2f}T')
print('\nPer-class (100 images each):')
for ci, cn in enumerate(CLASS_NAMES):
    bar = '#'*int(B_PC[ci]*10) + '-'*(10-int(B_PC[ci]*10))
    print(f'  {cn:<12} [{bar}] {B_PC[ci]:.0%}')

In [ ]:
# CELL 7 — Staged Embarrassment Training
model = build_model()
opt   = optim.Adam(model.parameters(), lr=BASE_LR)
crit  = nn.CrossEntropyLoss()

hist        = []
total_flops = 0.0
epoch_num   = 0
torch.cuda.reset_peak_memory_stats()
t_start = time.time()

for si, (sname, ps, pe, thresh, n_ep) in enumerate(STAGE_CONFIG):
    gamma  = GUILT_THRESHOLD * (1.0 - 0.4 * si / len(STAGE_CONFIG))
    loader = get_stage_loader(si)
    n_samp = len(loader.dataset)
    print(f'\n{"-"*58}')
    print(f'  {sname}  n={n_samp}  thresh={thresh:.0%}  gamma={gamma:.5f}')
    print(f'{"-"*58}')

    for ep in range(n_ep):
        model.train()
        E_sum  = np.zeros(N_CLASSES)
        C_sum  = np.zeros(N_CLASSES)
        sp_sum = nb = ls = cor = tot = 0

        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            opt.zero_grad()
            out  = model(imgs)
            loss = crit(out, labels)

            E_b, C_b = pc_embarrassment(out.detach(), labels)
            E_sum += E_b; C_sum += C_b

            for pg in opt.param_groups:
                pg['lr'] = BASE_LR * (1.0 + 2.0 * loss.item())

            loss.backward()
            sp = sparse_update(model, gamma)
            opt.step()

            ls  += loss.item() * labels.size(0)
            cor += (out.argmax(1) == labels).sum().item()
            tot += labels.size(0)
            sp_sum += sp; nb += 1
            total_flops += flops(sp, labels.size(0))

        avg_E  = E_sum / nb
        avg_C  = C_sum / nb
        avg_sp = sp_sum / nb

        # Both train and test accuracies
        full_acc = evaluate_full(model, full_test_loader)
        pc_acc, test100_acc = evaluate_100_per_class(model)
        epoch_num += 1

        row = {
            'epoch': epoch_num, 'stage': si+1, 'stage_name': sname,
            'train_acc': cor/tot, 'train_loss': ls/tot,
            'test_full_acc': full_acc,
            'test_100_acc': test100_acc,
            'sparsity': avg_sp,
            'flops_T': total_flops/1e12,
            'time_s': time.time()-t_start,
        }
        for ci, cn in enumerate(CLASS_NAMES):
            row[f'E_{cn}']   = avg_E[ci]
            row[f'C_{cn}']   = avg_C[ci]
            row[f'acc_{cn}'] = pc_acc[ci]
        hist.append(row)

        bar = '#'*int(full_acc*20) + '-'*(20-int(full_acc*20))
        e_s = ' '.join([f'{cn[:3]}:{avg_E[ci]:.2f}' for ci,cn in enumerate(CLASS_NAMES)])
        print(f'  Ep{ep+1:2d}/{n_ep} [{bar}] full={full_acc:.1%} t100={test100_acc:.1%} Sp={avg_sp:.0%}')
        print(f'          E: {e_s}')

    final_full = evaluate_full(model, full_test_loader)
    _, final_100 = evaluate_100_per_class(model)
    status = 'PASS' if final_100 >= thresh else 'FAIL'
    print(f'  {status}  full={final_full:.1%}  test100={final_100:.1%}  vs thresh={thresh:.0%}')

elapsed      = time.time() - t_start
hist_df      = pd.DataFrame(hist)
S_FULL_ACC   = evaluate_full(model, full_test_loader)
S_PC, S_100  = evaluate_100_per_class(model)

print(f'\nStaged done in {elapsed:.0f}s')
print(f'  Full test accuracy  : {S_FULL_ACC:.1%}')
print(f'  100-per-class test  : {S_100:.1%}')
print(f'  Total FLOPs         : {total_flops/1e12:.3f}T')
print('\nPer-class (100 images each):')
for ci, cn in enumerate(CLASS_NAMES):
    bar = '#'*int(S_PC[ci]*10) + '-'*(10-int(S_PC[ci]*10))
    diff = (S_PC[ci] - B_PC[ci])*100
    print(f'  {cn:<12} [{bar}] {S_PC[ci]:.0%}  ({diff:+.0f}% vs baseline)')

In [ ]:
# CELL 8 — Thermal Snapshots: What each model predicted on 100-per-class test
# Shows a grid of test images colored by correct/wrong prediction

def plot_thermal_snapshot(model, title, filename):
    all_imgs, all_preds, all_labels, all_probs = get_predictions_on_100(model)

    fig, axes = plt.subplots(10, 10, figsize=(20, 20))
    fig.patch.set_facecolor('#0a0a0a')

    idx = 0
    for row in range(10):
        for col in range(10):
            ax = axes[row, col]
            img = np.transpose(all_imgs[idx], (1,2,0))
            img = np.clip(img, 0, 1)
            ax.imshow(img, interpolation='nearest')
            correct = all_preds[idx] == all_labels[idx]
            conf    = all_probs[idx][all_preds[idx]]
            # Border color: green=correct, red=wrong, brightness=confidence
            border_c = '#00ff88' if correct else '#ff4444'
            for spine in ax.spines.values():
                spine.set_edgecolor(border_c)
                spine.set_linewidth(2.5 * conf)
            ax.set_xticks([]); ax.set_yticks([])
            if col == 0:
                ax.set_ylabel(CLASS_NAMES[row][:4], color='#cccccc', fontsize=7, rotation=0, labelpad=22)
            idx += 1

    fig.suptitle(
        f'{title}\nGreen border = correct  |  Red border = wrong  |  Border thickness = confidence',
        color='#cccccc', fontsize=12, fontweight='bold', y=1.01
    )
    plt.tight_layout(pad=0.3)
    plt.savefig(filename, dpi=120, bbox_inches='tight',
                facecolor='#0a0a0a', edgecolor='none')
    plt.show()
    print(f'Saved {filename}')

plot_thermal_snapshot(b_model, 'Baseline Model — 100 Test Images per Class', '/content/thermal_baseline.png')
plot_thermal_snapshot(model,   'Staged Model  — 100 Test Images per Class',  '/content/thermal_staged.png')

In [ ]:
# CELL 9 — Full Results Graphs
BG = '#0a0a0a'; CARD = '#111111'; GRID = '#1e1e1e'; TEXT = '#cccccc'
STAGED_C = '#c8b8ff'; BASE_C = '#ff9999'
CLS_C = ['#FFB3BA','#BAFFC9','#BAD4FF','#FFE4BA','#E8BAFF',
         '#B3FFE8','#FFBAF0','#D4FFBA','#BAE8FF','#FFF0BA']

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': CARD,
    'axes.edgecolor': GRID, 'axes.labelcolor': TEXT,
    'xtick.color': TEXT, 'ytick.color': TEXT, 'text.color': TEXT,
    'grid.color': GRID, 'grid.alpha': 0.3,
    'legend.facecolor': '#1a1a1a', 'legend.edgecolor': GRID,
    'font.family': 'monospace',
})

def sx(ax, title):
    ax.set_facecolor(CARD)
    ax.set_title(title, color=TEXT, fontsize=10, fontweight='bold', pad=8)
    ax.tick_params(colors='#666', labelsize=8)
    for sp in ax.spines.values(): sp.set_color(GRID)
    ax.grid(True)

pct  = plt.FuncFormatter(lambda y,_: f'{y:.0%}')
s_ep = hist_df['epoch'].values
b_ep = b_hist_df['epoch'].values

stage_starts, ep_c = [], 0
for *_, n_ep in STAGE_CONFIG:
    stage_starts.append(ep_c+1); ep_c += n_ep

def add_stages(ax):
    ylim = ax.get_ylim()
    for i, xs in enumerate(stage_starts):
        ax.axvline(xs, color='#2a2a2a', lw=1, ls='--')
        ax.text(xs+0.2, ylim[1]*0.96, f'S{i+1}', color='#444', fontsize=7)

fig = plt.figure(figsize=(22, 36))
gs  = gridspec.GridSpec(6, 2, figure=fig, hspace=0.55, wspace=0.35)

# 1 — Train vs Test accuracy (staged)
ax = fig.add_subplot(gs[0,0]); sx(ax, 'Train vs Test Accuracy (Staged)')
ax.plot(s_ep, hist_df['train_acc'],      color='#ffcc88', lw=2,   label='Train acc')
ax.plot(s_ep, hist_df['test_full_acc'],  color=STAGED_C,  lw=2.5, label='Test full (10k)')
ax.plot(s_ep, hist_df['test_100_acc'],   color='#88ff88', lw=2,   ls='--', label='Test 100/class')
add_stages(ax)
ax.set_xlabel('Epoch'); ax.yaxis.set_major_formatter(pct)
ax.legend(fontsize=8)

# 2 — Train vs Test accuracy (baseline)
ax = fig.add_subplot(gs[0,1]); sx(ax, 'Train vs Test Accuracy (Baseline)')
ax.plot(b_ep, b_hist_df['train_acc'],     color='#ffcc88', lw=2,   label='Train acc')
ax.plot(b_ep, b_hist_df['test_full_acc'], color=BASE_C,    lw=2.5, label='Test full (10k)')
ax.plot(b_ep, b_hist_df['test_100_acc'],  color='#ff8888', lw=2,   ls='--', label='Test 100/class')
ax.set_xlabel('Epoch'); ax.yaxis.set_major_formatter(pct)
ax.legend(fontsize=8)

# 3 — Head to head: test accuracy staged vs baseline
ax = fig.add_subplot(gs[1,0]); sx(ax, 'Test Accuracy — Staged vs Baseline')
ax.plot(b_ep, b_hist_df['test_full_acc'], color=BASE_C,   lw=2, ls='--', label=f'Baseline {B_FULL_ACC:.1%}')
ax.plot(s_ep, hist_df['test_full_acc'],   color=STAGED_C, lw=2.5, label=f'Staged {S_FULL_ACC:.1%}')
add_stages(ax)
ax.set_xlabel('Epoch'); ax.yaxis.set_major_formatter(pct)
ax.legend(fontsize=8)

# 4 — Per-class E
ax = fig.add_subplot(gs[1,1]); sx(ax, 'Per-Class Embarrassment E (Staged)')
for ci, cn in enumerate(CLASS_NAMES):
    ax.plot(s_ep, hist_df[f'E_{cn}'], color=CLS_C[ci], lw=1.5, label=cn)
add_stages(ax)
ax.set_xlabel('Epoch'); ax.set_ylabel('E')
ax.legend(fontsize=7, ncol=2)

# 5 — Per-class C
ax = fig.add_subplot(gs[2,0]); sx(ax, 'Per-Class Confidence C (Staged)')
for ci, cn in enumerate(CLASS_NAMES):
    ax.plot(s_ep, hist_df[f'C_{cn}'], color=CLS_C[ci], lw=1.5, label=cn)
add_stages(ax)
ax.set_xlabel('Epoch'); ax.set_ylabel('C')
ax.legend(fontsize=7, ncol=2)

# 6 — Sparsity
ax = fig.add_subplot(gs[2,1]); sx(ax, 'Sparse Updates — Fraction Frozen')
ax.fill_between(s_ep, hist_df['sparsity'], alpha=0.25, color='#b3d9ff')
ax.plot(s_ep, hist_df['sparsity'], color='#b3d9ff', lw=2)
ax.axhline(0.0, color=BASE_C, lw=1, ls=':', label='Baseline (0% frozen)')
add_stages(ax)
ax.set_xlabel('Epoch'); ax.yaxis.set_major_formatter(pct)
ax.legend(fontsize=8)

# 7 — FLOPs
ax = fig.add_subplot(gs[3,0]); sx(ax, 'Cumulative FLOPs')
ax.plot(b_ep, b_hist_df['flops_T'], color=BASE_C,   lw=2, ls='--', label=f'Baseline {b_flops/1e12:.1f}T')
ax.plot(s_ep, hist_df['flops_T'],   color=STAGED_C, lw=2.5, label=f'Staged {total_flops/1e12:.2f}T')
add_stages(ax)
ax.set_xlabel('Epoch'); ax.set_ylabel('FLOPs (T)')
ax.legend(fontsize=8)

# 8 — Time
ax = fig.add_subplot(gs[3,1]); sx(ax, 'Training Time')
ax.plot(b_ep, b_hist_df['time_s'], color=BASE_C,   lw=2, ls='--', label=f'Baseline {b_elapsed:.0f}s')
ax.plot(s_ep, hist_df['time_s'],   color=STAGED_C, lw=2.5, label=f'Staged {elapsed:.0f}s')
add_stages(ax)
ax.set_xlabel('Epoch'); ax.set_ylabel('Time (s)')
ax.legend(fontsize=8)

# 9 — Per-class accuracy bar (100 per class test) baseline vs staged
ax = fig.add_subplot(gs[4,:]); sx(ax, 'Per-Class Test Accuracy — 100 Images per Class')
x = np.arange(N_CLASSES); w = 0.35
ax.bar(x-w/2, B_PC, w, color=BASE_C,   alpha=0.85, label=f'Baseline avg={B_100:.1%}')
ax.bar(x+w/2, S_PC, w, color=STAGED_C, alpha=0.85, label=f'Staged  avg={S_100:.1%}')
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=20, ha='right', color=TEXT, fontsize=9)
ax.yaxis.set_major_formatter(pct)
ax.legend(fontsize=9)
for xi in range(N_CLASSES):
    diff = (S_PC[xi] - B_PC[xi])*100
    ax.text(xi+w/2, S_PC[xi]+0.01, f'{diff:+.0f}%', ha='center', fontsize=7, color=TEXT)

# 10 — Scatter: accuracy vs FLOPs saved
SPARSE_SYSTEMS = {
    'Magnitude Pruning': (0.920, 75.0),
    'RigL (Google)':     (0.928, 80.0),
    'Lottery Ticket':    (0.925, 85.0),
}
ax = fig.add_subplot(gs[5,:]); sx(ax, 'Accuracy vs FLOPs Saved — Your System vs Published Methods')
flops_saved_pct = (1 - total_flops/1e12/(b_flops/1e12)) * 100
systems = [
    ('Dense Baseline',    B_100,   0.0,            BASE_C),
    ('Magnitude Pruning', 0.920,   75.0,           '#88ffcc'),
    ('RigL (Google)',     0.928,   80.0,           '#ffaa88'),
    ('Lottery Ticket',    0.925,   85.0,           '#aaaaff'),
    ('Yours (Staged)',    S_100,   flops_saved_pct, STAGED_C),
]
for sname, acc, saved, sc in systems:
    ax.scatter(saved, acc, color=sc, s=350, zorder=5)
    ax.annotate(sname, (saved, acc), xytext=(6,6),
                textcoords='offset points', fontsize=9, color=sc)
ax.set_xlabel('FLOPs Saved vs Dense Baseline (%)')
ax.set_ylabel('Test Accuracy (100/class held-out)')
ax.yaxis.set_major_formatter(pct)
ax.text(0.02, 0.05, 'Top-right corner = best', transform=ax.transAxes, color='#555', fontsize=9)

fig.suptitle(
    'Staged Embarrassment Learning — CIFAR-10 All 10 Classes\n'
    f'Baseline: {BASELINE_EPOCHS} epochs  |  Staged: {sum(s[4] for s in STAGE_CONFIG)} epochs  |  '
    f'Test: 100 images per class held-out\n'
    f'γ={GUILT_THRESHOLD:.5f}  T={TEMPERATURE}  {SAMPLES_PER_CLASS} samples/class/stage',
    fontsize=11, fontweight='bold', y=1.01, color=TEXT
)
plt.savefig('/content/results_final.png', dpi=130,
            bbox_inches='tight', facecolor=BG, edgecolor='none')
plt.show()
print('Saved results_final.png')

In [ ]:
# CELL 10 — Summary and Download
from google.colab import files

flops_saved = (1 - total_flops/1e12/(b_flops/1e12)) * 100
time_saved  = (1 - elapsed/b_elapsed) * 100

print('='*68)
print('  FINAL RESULTS — SAME HARDWARE, HELD-OUT TEST SET (100/CLASS)')
print('='*68)
print(f'  {"System":<22} {"Full Test":>10} {"100/class":>10} {"FLOPs":>8} {"Time":>8}')
print('-'*68)
print(f'  {"Baseline (30ep)":<22} {B_FULL_ACC:>9.1%} {B_100:>9.1%} {b_flops/1e12:>7.1f}T {b_elapsed:>7.0f}s')
print(f'  {"Staged (50ep)":<22} {S_FULL_ACC:>9.1%} {S_100:>9.1%} {total_flops/1e12:>7.3f}T {elapsed:>7.0f}s')
print(f'  {"Savings":<22} {(S_FULL_ACC-B_FULL_ACC)*100:>+9.1f}% {(S_100-B_100)*100:>+9.1f}% {flops_saved:>+7.0f}% {time_saved:>+7.0f}%')
print('='*68)
print('\nPer-class accuracy on 100 held-out images each:')
print(f'  {"Class":<12}  {"Baseline":>8}  {"Staged":>8}  {"Diff":>6}')
print('-'*44)
for ci, cn in enumerate(CLASS_NAMES):
    diff = (S_PC[ci] - B_PC[ci]) * 100
    print(f'  {cn:<12}  {B_PC[ci]:>7.0%}  {S_PC[ci]:>7.0%}  {diff:>+5.0f}%')

hist_df.to_csv('/content/staged_hist.csv',   index=False)
b_hist_df.to_csv('/content/baseline_hist.csv', index=False)
for f in ['/content/results_final.png',
          '/content/thermal_baseline.png',
          '/content/thermal_staged.png',
          '/content/staged_hist.csv',
          '/content/baseline_hist.csv']:
    files.download(f)
print('\nAll files downloaded')